# 02 · Prediccion de recompra

Ruta de continuacion con las seis fases de CRISP-DM. Los CSV se leen directamente desde el repositorio.

## 1. Comprension del negocio

Estimaremos si un cliente con compras previas vuelve a comprar durante una ventana futura de 90 dias. Esto puede orientar analisis de retencion, pero no explica causas ni justifica decisiones automatizadas sobre personas.

## 2. Comprension de los datos y EDA

Usamos pedidos, clientes, lineas y productos. `customer_unique_id` solo permite agrupar compras; no se usa como predictor ni aparece en la demo. La etiqueta requiere historia y una ventana futura completa. Kaufman et al. ([2012](https://doi.org/10.1145/2382577.2382579)) describen fuga de informacion; Roberts et al. ([2017](https://doi.org/10.1111/ecog.02881)) revisan particiones que respetan la estructura temporal. La aplicacion concreta debe corresponder al momento real de decision.


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"
def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(f"No se pudo cargar {filename} desde {url}. Compruebe la conexion a internet.") from error
orders = load_dataset_csv("olist_orders_dataset.csv")
customers = load_dataset_csv("olist_customers_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
print(f"Pedidos: {len(orders):,}; clientes unicos anonimizados: {customers['customer_unique_id'].nunique():,}")


### EDA de tablas fuente

Revisamos estados y volumen de pedidos por mes antes de definir la poblacion. La etiqueta y las caracteristicas se restringen a compras entregadas: pedidos cancelados o no entregados no cuentan como recompra observada.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
print("Estados registrados:")
display(orders["order_status"].value_counts(dropna=False).to_frame("pedidos"))
monthly_orders = orders.dropna(subset=["order_purchase_timestamp"]).set_index("order_purchase_timestamp").resample("MS").size()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
orders["order_status"].value_counts().plot.bar(ax=axes[0], color="slateblue")
axes[0].set_title("Pedidos por estado")
axes[0].set_ylabel("Pedidos")
monthly_orders.plot(ax=axes[1], color="darkorange")
axes[1].set_title("Pedidos por mes")
axes[1].set_ylabel("Pedidos")
axes[1].set_xlabel("Mes de compra")
plt.tight_layout()
plt.show()
print(f"Cobertura: {orders['order_purchase_timestamp'].min().date()} a {orders['order_purchase_timestamp'].max().date()}; fechas ausentes: {orders['order_purchase_timestamp'].isna().sum():,}")


## 3. Preparacion: cohortes temporales sin fuga

Cada fotografia usa 180 dias de historia y una etiqueta de 90 dias futuros. Crearemos varias fechas de corte separadas por unos tres meses. Ordenamos esas cohortes: primeras para entrenamiento, siguientes para validacion y ultimas para prueba. Una misma persona puede aparecer en mas de una fecha, como ocurriria al predecir repetidamente para clientes conocidos; en cada fila los predictores siguen siendo anteriores al corte y la etiqueta posterior.


In [ ]:
delivered = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner")
delivered = delivered.loc[
    delivered["order_status"].eq("delivered")
    & delivered["order_purchase_timestamp"].notna()
    & delivered["customer_unique_id"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con fecha y cliente validos.")
window_days, history_days = 90, 180
first_cutoff = (delivered["order_purchase_timestamp"].min() + pd.Timedelta(days=history_days)).to_period("M").to_timestamp() + pd.offsets.MonthBegin(1)
last_cutoff = (delivered["order_purchase_timestamp"].max() - pd.Timedelta(days=window_days)).normalize()
cutoffs = pd.date_range(first_cutoff, last_cutoff, freq="3MS")
if len(cutoffs) < 5:
    raise ValueError("Se requieren al menos cinco cohortes temporales completas para comparar train, validacion y test.")
rows = []
for cutoff in cutoffs:
    history_start = cutoff - pd.Timedelta(days=history_days)
    future_end = cutoff + pd.Timedelta(days=window_days)
    history = delivered.loc[
        delivered["order_purchase_timestamp"].ge(history_start)
        & delivered["order_purchase_timestamp"].lt(cutoff)
    ]
    future = delivered.loc[
        delivered["order_purchase_timestamp"].ge(cutoff)
        & delivered["order_purchase_timestamp"].lt(future_end)
    ]
    if history.empty or future.empty:
        continue
    cohort = history.groupby("customer_unique_id").agg(
        frequency=("order_id", "nunique"),
        last_purchase=("order_purchase_timestamp", "max"),
    )
    cohort["recency_days"] = (cutoff - cohort.pop("last_purchase")).dt.total_seconds() / 86400
    history_orders = history[["order_id", "customer_unique_id"]].drop_duplicates()
    history_items = items.merge(history_orders, on="order_id", how="inner").merge(
        products[["product_id", "product_category_name"]], on="product_id", how="left"
    )
    item_features = history_items.groupby("customer_unique_id").agg(
        past_spend_brl=("price", "sum"),
        product_variety=("product_id", "nunique"),
        category_variety=("product_category_name", "nunique"),
    )
    cohort = cohort.join(item_features, how="left").fillna(0)
    future_customers = set(future["customer_unique_id"])
    cohort["repurchased"] = cohort.index.isin(future_customers).astype(int)
    cohort["snapshot_date"] = cutoff
    rows.append(cohort.reset_index(drop=True))

dataset = pd.concat(rows, ignore_index=True)
feature_columns = ["frequency", "recency_days", "past_spend_brl", "product_variety", "category_variety"]
if dataset["snapshot_date"].nunique() < 5:
    raise ValueError("No hay suficientes cohortes completas despues de preparar las ventanas.")
print(f"Fotografias temporales: {dataset['snapshot_date'].nunique()}; rango {dataset['snapshot_date'].min().date()} a {dataset['snapshot_date'].max().date()}; filas cliente-cohorte: {len(dataset):,}")


### EDA de las etiquetas y predictores

La proporción positiva por cohorte muestra si la recompra cambia con el calendario; la distribución de predictores muestra asimetría. Si pocos clientes recompran, una predicción mayoritaria puede alcanzar accuracy engañosa. Usaremos precisión, exhaustividad, F1 y PR-AUC, además de observar la prevalencia.


In [ ]:
class_balance = dataset.groupby("snapshot_date")["repurchased"].agg(["count", "mean"]).rename(columns={"mean": "prevalencia_recompra"})
display(class_balance.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_balance["prevalencia_recompra"].plot(marker="o", ax=axes[0], color="darkorange")
axes[0].set_title("Prevalencia futura por cohorte")
axes[0].set_ylabel("Proporcion de clientes con recompra")
axes[0].set_xlabel("Fecha de corte")
for column in ["frequency", "recency_days", "past_spend_brl"]:
    axes[1].hist(np.log1p(dataset[column].clip(lower=0)), bins=35, alpha=0.55, label=column)
axes[1].set_title("Predictores históricos — log(1+x)")
axes[1].set_xlabel("Valor transformado")
axes[1].set_ylabel("Filas cliente-cohorte")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Prevalencia global en las cohortes: {dataset['repurchased'].mean():.1%}")
display(dataset[feature_columns].describe().T.round(2))


Los importes y conteos suelen tener escalas y colas distintas. La regresión logística se estandariza; los árboles no requieren escalado. La imputación se ubica dentro de cada pipeline para no calcular estadísticos usando cohortes futuras.

## 4. Modelado: comparar y seleccionar

Dummy es la referencia de clase mayoritaria. Comparamos regresión logística balanceada, Random Forest balanceado e HistGradientBoosting balanceado. El modelo lineal y los dos ensambles ofrecen supuestos distintos. Train, validación y test se separan por fecha de corte, no al azar; seleccionamos por F1 de validación. Saito & Rehmsmeier ([2015](https://doi.org/10.1371/journal.pone.0118432)) muestran que precision-recall puede ser más informativa que ROC con clases desbalanceadas; aquí añadimos PR-AUC, sin inventar un umbral de negocio.


In [ ]:
cohort_dates = np.sort(dataset["snapshot_date"].unique())
train_end = cohort_dates[max(1, int(len(cohort_dates) * 0.60) - 1)]
validation_end = cohort_dates[max(2, int(len(cohort_dates) * 0.80) - 1)]
train_rows = dataset.loc[dataset["snapshot_date"] <= train_end]
validation_rows = dataset.loc[(dataset["snapshot_date"] > train_end) & (dataset["snapshot_date"] <= validation_end)]
test_rows = dataset.loc[dataset["snapshot_date"] > validation_end]
if min(len(train_rows), len(validation_rows), len(test_rows)) == 0:
    raise ValueError("La division temporal dejo vacio train, validacion o test.")
X_train, y_train = train_rows[feature_columns], train_rows["repurchased"]
X_validation, y_validation = validation_rows[feature_columns], validation_rows["repurchased"]
X_train_validation = pd.concat([X_train, X_validation], ignore_index=True)
y_train_validation = pd.concat([y_train, y_validation], ignore_index=True)
X_test, y_test = test_rows[feature_columns], test_rows["repurchased"]
for label, target in [("train", y_train), ("validacion", y_validation), ("test", y_test)]:
    if target.nunique() < 2:
        raise ValueError(f"La particion {label} no contiene ambas clases; revise cohortes disponibles.")
print(f"Train hasta {pd.Timestamp(train_end).date()}; validacion hasta {pd.Timestamp(validation_end).date()}; test en {test_rows['snapshot_date'].min().date()}..{test_rows['snapshot_date'].max().date()}")
models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=180, min_samples_leaf=4, class_weight="balanced_subsample", random_state=42, n_jobs=-1)),
    "HistGradientBoosting": make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(max_iter=140, l2_regularization=1.0, class_weight="balanced", random_state=42)),
}
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
for estimator in models.values():
    estimator.fit(X_train, y_train)

def positive_probability(estimator, data):
    classes = estimator.classes_
    if 1 not in classes:
        return np.zeros(len(data))
    return estimator.predict_proba(data)[:, list(classes).index(1)]

validation_predictions = {"Referencia mayoritaria": baseline.predict(X_validation)}
validation_probabilities = {"Referencia mayoritaria": positive_probability(baseline, X_validation)}
for name, estimator in models.items():
    validation_predictions[name] = estimator.predict(X_validation)
    validation_probabilities[name] = positive_probability(estimator, X_validation)
validation_results = pd.DataFrame([
    {"modelo": name, "precision": precision_score(y_validation, prediction, zero_division=0),
     "exhaustividad": recall_score(y_validation, prediction, zero_division=0),
     "F1": f1_score(y_validation, prediction, zero_division=0),
     "PR-AUC": average_precision_score(y_validation, validation_probabilities[name])}
    for name, prediction in validation_predictions.items()
]).sort_values(["F1", "precision"], ascending=False)
display(validation_results.round(3))
best_name = validation_results.iloc[0]["modelo"]
print(f"Seleccion en validacion temporal por F1: {best_name}")


## 5. Evaluacion temporal

Reajustamos el ganador con train+validacion y lo evaluamos una vez en las cohortes posteriores de test. Comparamos con la referencia, mostramos matriz de confusion, precision, exhaustividad, F1 y PR-AUC. La estimacion describe esas cohortes, no una garantia para periodos futuros.


In [ ]:
if best_name == "Referencia mayoritaria":
    best_model = DummyClassifier(strategy="most_frequent").fit(X_train_validation, y_train_validation)
else:
    best_model = models[best_name]
    best_model.fit(X_train_validation, y_train_validation)
baseline_test = DummyClassifier(strategy="most_frequent").fit(X_train_validation, y_train_validation)
test_prediction = best_model.predict(X_test)
test_probability = positive_probability(best_model, X_test)
baseline_prediction = baseline_test.predict(X_test)
test_results = pd.DataFrame([
    {"modelo": "Referencia mayoritaria", "precision": precision_score(y_test, baseline_prediction, zero_division=0), "exhaustividad": recall_score(y_test, baseline_prediction, zero_division=0), "F1": f1_score(y_test, baseline_prediction, zero_division=0), "PR-AUC": average_precision_score(y_test, positive_probability(baseline_test, X_test))},
    {"modelo": best_name, "precision": precision_score(y_test, test_prediction, zero_division=0), "exhaustividad": recall_score(y_test, test_prediction, zero_division=0), "F1": f1_score(y_test, test_prediction, zero_division=0), "PR-AUC": average_precision_score(y_test, test_probability)},
])
display(test_results.round(3))
ConfusionMatrixDisplay(confusion_matrix(y_test, test_prediction, labels=[0, 1]), display_labels=["No recompra", "Recompra"]).plot(cmap="Blues")
plt.title(f"Test temporal — modelo elegido: {best_name}")
plt.tight_layout()
plt.show()


¿Qué errores serían más costosos para una iniciativa real? La respuesta afecta al umbral y a la métrica; aquí no se conocen esos costos y no se recomienda actuar sobre predicciones individuales.

## 6. Despliegue: demo Gradio temporal

La interfaz recibe agregados ilustrativos, no identificadores. Usa el ganador seleccionado en validacion temporal. En Colab es temporal y no es un despliegue de produccion ni una recomendacion para clientes reales.


In [ ]:
import gradio as gr

def repurchase_demo(frequency, recency_days, past_spend_brl, product_variety, category_variety):
    example = pd.DataFrame([[frequency, recency_days, past_spend_brl, product_variety, category_variety]], columns=feature_columns)
    probability = float(positive_probability(best_model, example)[0])
    predicted = int(best_model.predict(example)[0])
    label = "recompra" if predicted else "sin recompra"
    return f"Modelo: {best_name}. Clase exploratoria: {label}; probabilidad estimada en ventana de 90 dias: {probability:.1%}."

demo = gr.Interface(
    fn=repurchase_demo,
    inputs=[
        gr.Number(value=2, minimum=1, label="Pedidos historicos"),
        gr.Number(value=30, minimum=0, label="Dias desde ultima compra"),
        gr.Number(value=150, minimum=0, label="Suma de precios (BRL)"),
        gr.Number(value=2, minimum=1, label="Productos distintos"),
        gr.Number(value=2, minimum=1, label="Categorias distintas"),
    ],
    outputs=gr.Textbox(label="Resultado educativo"),
    title="Demo de recompra",
    description=f"Ganador por F1 en validacion temporal: {best_name}. No usar para decisiones reales.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)
